# Housing Strand - Data Explore

**MultimodalAI'26 - Housing Demo**

This notebook introduces the demo subset and helps participants understand what data they are working with before preprocessing and model training.

**Objective**
- Load modality-specific CSV files.
- Merge them into one daily property table.
- Create the `cold_risk` label used throughout the strand.
- Review missingness patterns (especially MNAR-relevant fields like CO2 and survey).
- Save a merged dataset for the next notebook.

**Hackathon context reminder**
- Required deliverables: solution, OMAIB pathway, housing benchmark card, evidence dashboard, and option-specific deliverable.
- Role split: Builder, Evidence Analyst, Governance Lead.

**Starter-kit boundary**
- This notebook builds evidence inputs only.
- Participants must still produce and justify final deliverables themselves.

**Prerequisite**
- `demo/data/raw/` files must exist.

**Sections**
1. Setup paths and imports.
2. Load and merge modality data.
3. Inspect shape, property count, and cold-risk prevalence.
4. Compute missingness and save `merged_demo.csv`.

## Section 1 - Setup and Paths

This cell imports libraries and resolves `DEMO_ROOT` so the notebook can run whether you open it from repository root or inside `demo/`.

Expected outcome:
- `RAW_DIR` points to `demo/data/raw`
- `PROCESSED_DIR` points to `demo/data/processed`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    DEMO_ROOT = cwd
elif (cwd / "demo" / "src").exists():
    DEMO_ROOT = cwd / "demo"
else:
    DEMO_ROOT = cwd.parent

sys.path.append(str(DEMO_ROOT))
RAW_DIR = DEMO_ROOT / "data" / "raw"
PROCESSED_DIR = DEMO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR

## Section 2 - Load and Merge Modalities

This cell loads metadata + modality tables and merges them into one daily table keyed by:
- `reference`, `year`, `month`, `day`

It also creates the strand target label:
- `cold_risk = (avgTemperature < 19.0)`

In [ ]:
properties = pd.read_csv(RAW_DIR / "properties.csv")
env = pd.read_csv(RAW_DIR / "indoor_environment_daily.csv")
energy = pd.read_csv(RAW_DIR / "energy_noise_daily.csv")
survey = pd.read_csv(RAW_DIR / "resident_feedback_daily.csv")

key_cols = ["reference", "year", "month", "day"]
merged = env.merge(energy, on=key_cols, how="left").merge(survey, on=key_cols, how="left")
merged = merged.merge(properties, on="reference", how="left")
merged["date"] = pd.to_datetime(merged[["year", "month", "day"]])
merged = merged.sort_values(["reference", "date"]).reset_index(drop=True)
merged["cold_risk"] = (merged["avgTemperature"] < 19.0).astype(int)

print("rows:", len(merged), "| properties:", merged["reference"].nunique())
print("cold_risk_rate:", round(merged["cold_risk"].mean(), 3))
merged.head()

## Section 3 - Data Quality Snapshot

This cell computes missingness across core features, draws a quick bar chart, and saves:
- `demo/data/processed/merged_demo.csv`

Use this output in Notebook 02 for preprocessing and leakage-safe split.

In [ ]:
missing = merged[["avgTemperature", "avgHumidity", "avgCo2", "smart_meter_kwh", "noise_db", "survey_score"]].isna().mean().sort_values(ascending=False)
display(missing.to_frame("missing_rate"))

plt.figure(figsize=(8, 3))
sns.barplot(x=missing.index, y=missing.values)
plt.xticks(rotation=30, ha="right")
plt.ylabel("missing rate")
plt.title("Demo subset missingness")
plt.tight_layout()
plt.show()

merged.to_csv(PROCESSED_DIR / "merged_demo.csv", index=False)
print("saved:", PROCESSED_DIR / "merged_demo.csv")